1. ilk önce togg bulma modeli ile aracın togg olup olmadığına bakalım 

In [1]:
!pip install roboflow ultralytics -q
import cv2
import os
from ultralytics import YOLO

# ==========================================
# 1. DOSYA YOLLARINI KESİN VE DOĞRU GİR KANKA
# ==========================================
# UYARI: Klasör yolu değil, sonu .mp4 ve .pt ile biten tam yolları yazmalısın!
VIDEO_YOLU = '/kaggle/input/datasets/tbitakyandek/datasettest/WhatsApp Video 2026-06-09 at 10.18.22 AM (1).mp4' 
CIKTI_YOLU = '/kaggle/working/akilli_tespit_sonucu.mp4'

# Yolların sonuna /best.pt eklendi!
TOGG_MODEL_YOLU = '/kaggle/input/models/tbitakyandek/toggmodel/pytorch/default/1/toggModel.pt'
ACIL_MODEL_YOLU = '/kaggle/input/models/tbitakyandek/acilaratespiti/pytorch/default/1/emergencyModel.pt'

if not os.path.exists(VIDEO_YOLU):
    raise FileNotFoundError(f"🚨 HATA: Video bulunamadı! Yol hatalı: {VIDEO_YOLU}")

print("🔄 Modeller GPU/CPU'ya yükleniyor...")
model_togg = YOLO(TOGG_MODEL_YOLU, task='detect')
model_acil = YOLO(ACIL_MODEL_YOLU, task='detect')

# ==========================================
# 2. VİDEO VE DİZİ AYARLARI
# ==========================================
cap = cv2.VideoCapture(VIDEO_YOLU)
if not cap.isOpened():
    raise ValueError("🚨 HATA: Video okunamadı!")

fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(CIKTI_YOLU, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# İstediğin o kayıt dizisi
olay_kayitlari = []

print(f"🎬 Akıllı Video Analizi Başladı... (Çözünürlük: {w}x{h})\n")

# ==========================================
# 3. KADEMELİ KARE İŞLEME MANTIĞI (CASCADING)
# ==========================================
kare_no = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    kare_no += 1
    
    # ADIM 1: Önce Togg modelini koştur
    sonuc_togg = model_togg.predict(frame, conf=0.25, verbose=False)
    
    # Eğer o karede en az 1 tane Togg kutusu (boxes) varsa:
    if len(sonuc_togg[0].boxes) > 0:
        # Kutuyu çizdir
        frame = sonuc_togg[0].plot(img=frame)
        
        # Diziye kaydet ve konsola yaz
        mesaj = f"Kare {kare_no}: Togg Tespit Edildi!"
        print(f"🚗 {mesaj}")
        olay_kayitlari.append({"kare": kare_no, "tespit": "Togg"})
        
    # ADIM 2: Togg YOKSA (else mantığı), Acil Araç modelini koştur
    else:
        sonuc_acil = model_acil.predict(frame, conf=0.25, verbose=False)
        
        # Eğer Acil Araç varsa:
        if len(sonuc_acil[0].boxes) > 0:
            frame = sonuc_acil[0].plot(img=frame)
            
            mesaj = f"Kare {kare_no}: Acil Araç (Ambulans/İtfaiye) Tespit Edildi!"
            print(f"🚑 {mesaj}")
            olay_kayitlari.append({"kare": kare_no, "tespit": "Acil Araç"})
            
    # İşlenmiş/Çizilmiş kareyi çıktı videosuna yaz
    out.write(frame)

# ==========================================
# 4. TEMİZLİK VE ÇIKTI
# ==========================================
cap.release()
out.release()

print("\n" + "="*40)
print(f"🎉 İşlem Tamam! Akıllı video şuraya kaydedildi: {CIKTI_YOLU}")
print(f"📊 Toplam {len(olay_kayitlari)} adet olay diziye kaydedildi.")
print("="*40)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 

In [2]:
import cv2
import os
from ultralytics import YOLO

# ==========================================
# 1. DOSYA YOLLARINI KESİN VE DOĞRU GİR KANKA
# ==========================================
VIDEO_YOLU = '/kaggle/input/datasets/tbitakyandek/datasettest/WhatsApp Video 2026-06-09 at 10.18.22 AM (1).mp4'
CIKTI_YOLU = '/kaggle/working/plaka_tespit_sonucu11.mp4'

PLAKA_TESPIT_MODELI_YOLU = '/kaggle/input/models/tbitakyandek/plakakonumubulma/pytorch/default/1/best (3).pt'
PLAKA_OKUMA_MODELI_YOLU = '/kaggle/input/models/tbitakyandek/plakakarakterokum/pytorch/default/1/best_karakter_okuma.pt' 

if not os.path.exists(VIDEO_YOLU):
    raise FileNotFoundError(f"🚨 HATA: Video bulunamadı! Yol hatalı: {VIDEO_YOLU}")

print("🔄 Modeller GPU/CPU'ya yükleniyor...")
model_tespit = YOLO(PLAKA_TESPIT_MODELI_YOLU, task='detect')
model_okuma = YOLO(PLAKA_OKUMA_MODELI_YOLU, task='detect')

# ==========================================
# 2. VİDEO VE DİZİ AYARLARI
# ==========================================
cap = cv2.VideoCapture(VIDEO_YOLU)
if not cap.isOpened():
    raise ValueError("🚨 HATA: Video okunamadı!")

fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(CIKTI_YOLU, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# KANKA BURASI YENİ: Plakaların istatistiğini tutacağımız sözlük (Frekans ve Güven Skoru)
plaka_istatistik = {} 

print(f"🎬 Akıllı Plaka Analizi Başladı... (Çözünürlük: {w}x{h})")
print("⏳ Lütfen bekleyin, video arka planda sessizce taranıyor... \n")

# ==========================================
# 3. KARE KARE İŞLEME MANTIĞI
# ==========================================
kare_no = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    kare_no += 1

    sonuc_tespit = model_tespit.predict(frame, conf=0.25, verbose=False)

    if len(sonuc_tespit[0].boxes) > 0:
        for box in sonuc_tespit[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            cropped_plaka = frame[y1:y2, x1:x2]

            sonuc_okuma = model_okuma.predict(cropped_plaka, conf=0.25, verbose=False)

            if len(sonuc_okuma[0].boxes) > 0:
                karakterler = []
                for ocr_box in sonuc_okuma[0].boxes:
                    x_koordinati = float(ocr_box.xyxy[0][0]) 
                    class_id = int(ocr_box.cls[0])
                    harf = model_okuma.names[class_id]
                    # YENİ: Karakterin okunma güven skorunu (confidence) alıyoruz
                    guven = float(ocr_box.conf[0]) 
                    
                    karakterler.append((x_koordinati, harf, guven))
                
                # Karakterleri soldan sağa sırala
                karakterler.sort(key=lambda x: x[0])
                plaka_metni = "".join([k[1] for k in karakterler])
                
                # YENİ: Bu plakanın ortalama okunma güvenini hesapla
                ortalama_guven = sum([k[2] for k in karakterler]) / len(karakterler) if len(karakterler) > 0 else 0

                # Çizim
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"{plaka_metni}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

                # YENİ: Filtreleme ve İstatistik (5 karakterden kısa saçma metinleri çöpe atıyoruz)
                if len(plaka_metni) >= 5:
                    if plaka_metni not in plaka_istatistik:
                        plaka_istatistik[plaka_metni] = {"frekans": 0, "max_guven": 0.0}
                    
                    # O plakanın görülme sayısını artır ve en yüksek güven skorunu güncelle
                    plaka_istatistik[plaka_metni]["frekans"] += 1
                    if ortalama_guven > plaka_istatistik[plaka_metni]["max_guven"]:
                        plaka_istatistik[plaka_metni]["max_guven"] = ortalama_guven

    out.write(frame)

# ==========================================
# 4. TEMİZLİK VE EN İYİ PLAKAYI SEÇME
# ==========================================
cap.release()
out.release()

print("\n" + "="*50)
print(f"🎉 İşlem Tamam! İşlenmiş video kaydedildi: {CIKTI_YOLU}")
print("="*50)

# KANKA İŞTE BEKLEDİĞİN O TEK VE EN GÜVENİLİR SONUÇ BURADA:
print("\n🚀 VİDEO BOYUNCA TESPİT EDİLEN EN İYİ/EN GÜVENİLİR PLAKA:")

if len(plaka_istatistik) > 0:
    # Hem videoda en çok görülen (frekans) hem de güven skoru en yüksek olanı birinci seçiyoruz
    en_iyi_plaka = max(plaka_istatistik.keys(), key=lambda k: (plaka_istatistik[k]["frekans"], plaka_istatistik[k]["max_guven"]))
    detay = plaka_istatistik[en_iyi_plaka]
    
    print(f"🪪 TESPİT EDİLEN PLAKA : {en_iyi_plaka}")
    print(f"📊 Görülme Sayısı     : {detay['frekans']} farklı karede doğrulandı")
    print(f"🎯 Maksimum Güven     : %{detay['max_guven']*100:.2f}")
else:
    print("❌ Videoda geçerli/düzgün bir plaka okunamadı.")
print("="*50)

🔄 Modeller GPU/CPU'ya yükleniyor...
🎬 Akıllı Plaka Analizi Başladı... (Çözünürlük: 3840x2160)
⏳ Lütfen bekleyin, video arka planda sessizce taranıyor... 


🎉 İşlem Tamam! İşlenmiş video kaydedildi: /kaggle/working/plaka_tespit_sonucu11.mp4

🚀 VİDEO BOYUNCA TESPİT EDİLEN EN İYİ/EN GÜVENİLİR PLAKA:
🪪 TESPİT EDİLEN PLAKA : 34TC8532
📊 Görülme Sayısı     : 48 farklı karede doğrulandı
🎯 Maksimum Güven     : %91.80


In [3]:
import cv2
import os
from ultralytics import YOLO

# ==========================================
# 1. DOSYA VE MODEL YOLLARI
# ==========================================
VIDEO_YOLU = '/kaggle/input/datasets/tbitakyandek/datasettest/WhatsApp Video 2026-06-09 at 10.18.22 AM (1).mp4'
CIKTI_YOLU = '/kaggle/working/on_cam_sol_surucu_sonuc5563.mp4'

# Ön camı bulan kendi özel modelin
ONCAM_TESPIT_MODELI_YOLU = '/kaggle/input/models/tbitakyandek/oncambulmamodeli/pytorch/default/1/en_iyi_surucu_tespit_modeli (2).pt' 
# Standart COCO Modeli (person=0 sınıfı için)
COCO_MODEL_YOLU = 'yolov8n.pt' 

if not os.path.exists(VIDEO_YOLU):
    raise FileNotFoundError(f"🚨 HATA: Video bulunamadı: {VIDEO_YOLU}")

print("🔄 Modeller entegrasyon için yükleniyor...")
model_oncam = YOLO(ONCAM_TESPIT_MODELI_YOLU, task='detect')
model_coco = YOLO(COCO_MODEL_YOLU, task='detect')

# ==========================================
# 2. VİDEO VE EN İYİ TESPİT HAFIZASI
# ==========================================
cap = cv2.VideoCapture(VIDEO_YOLU)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(CIKTI_YOLU, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# En yüksek güvenli "en soldaki" sürücü kaydı
en_iyi_surucu_kaydi = {
    "guven_skoru": 0.0,
    "kare_no": 0
}

print("\n🎯 ÖN CAM İÇİ - EN SOLDAKİ KİŞİ FİLTRE TESTİ BAŞLADI...")
print("⏳ Büyük resimdeki diğer insanlar eleniyor, lütfen bekleyin...\n")

# ==========================================
# 3. KARE KARE İŞLEME MANTIĞI
# ==========================================
kare_no = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    kare_no += 1

    # ADIM 1: Önce senin modelle araç ön camını buluyoruz
    sonuc_oncam = model_oncam.predict(frame, conf=0.25, verbose=False)

    if len(sonuc_oncam[0].boxes) > 0:
        for box in sonuc_oncam[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cam_guven = float(box.conf[0])
            cam_class_id = int(box.cls[0])
            cam_etiket_adi = model_oncam.names[cam_class_id].upper() # Modelindeki etiket ismi (örn: ON_CAM)
            
            # 🔴 İSTEDİĞİN EKLEME: Ön camın kendisine de kutu ve etiket çiziyoruz kanka
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 2) # Ön camı turuncu çizelim
            cv2.putText(frame, f"{cam_etiket_adi} (%{cam_guven*100:.1f})", (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
            
            # Arama alanını sadece ÖN CAM KIRPINTISI yapıyoruz
            cropped_oncam = frame[y1:y2, x1:x2]

            # ADIM 2: Sadece bu kırpıntının içinde COCO modelini koşturuyoruz
            sonuc_coco = model_coco.predict(cropped_oncam, conf=0.25, verbose=False)

            if len(sonuc_coco[0].boxes) > 0:
                bulunan_insanlar = []

                for c_box in sonuc_coco[0].boxes:
                    class_id = int(c_box.cls[0])
                    
                    # COCO veri setinde 0 numaralı sınıf 'person' (insan) demektir
                    if class_id == 0:
                        guven = float(c_box.conf[0])
                        sx1, sy1, sx2, sy2 = map(int, c_box.xyxy[0])
                        
                        bulunan_insanlar.append({
                            "koordinat": (sx1, sy1, sx2, sy2),
                            "sol_x": sx1, # Soldaki kişiyi bulmak için referans noktamız
                            "guven": guven
                        })

                # Ön cam içindeki insanlardan EN SOLDAKİNİ seçiyoruz
                if bulunan_insanlar:
                    # İnsanları ön cam içindeki sol kenar konumuna (sol_x) göre sırala
                    bulunan_insanlar.sort(key=lambda x: x["sol_x"])
                    
                    # Listenin en başındaki (0. eleman) en soldaki kişidir, yani gerçek sürücü!
                    gercek_surucu = bulunan_insanlar[0]

                    # ==========================================
                    # ANA VİDEO ÜZERİNE SÜRÜCÜ ÇİZİMİ
                    # ==========================================
                    sx1, sy1, sx2, sy2 = gercek_surucu["koordinat"]
                    
                    # Kırpık camdaki yerel koordinatları ana video koordinatlarına dönüştür
                    ana_x1, ana_y1 = x1 + sx1, y1 + sy1
                    ana_x2, ana_y2 = x1 + sx2, y1 + sy2

                    # Sürücüyü yeşil kutuya al, sağdaki yolcular çizilmeyecek bile!
                    cv2.rectangle(frame, (ana_x1, ana_y1), (ana_x2, ana_y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"SURUCU (SOL) %{gercek_surucu['guven']*100:.1f}", (ana_x1, ana_y1 - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                    # Video genelinde bu şartı sağlayan en yüksek güvenli kareyi hafızaya yaz
                    if gercek_surucu["guven"] > en_iyi_surucu_kaydi["guven_skoru"]:
                        en_iyi_surucu_kaydi["guven_skoru"] = gercek_surucu["guven"]
                        en_iyi_surucu_kaydi["kare_no"] = kare_no

    out.write(frame)

cap.release()
out.release()

# ==========================================
# 4. KONSOL ÇIKTI RAPORU
# ==========================================
print("=" * 60)
print("🏁 ÖN CAM İÇİ GEOMETRİK ANALİZ TAMAMLANDI!")
print("=" * 60)

if en_iyi_surucu_kaydi["guven_skoru"] > 0:
    print("\n🏆 ÖN CAM KIRPINTISINDAKİ EN SOLDAN SÜRÜCÜ TESPİT SONUCU:")
    print("-" * 60)
    print(f" 🎯 Hedef Filtre      : Ön Cam İçi -> En Soldaki Person (Sürücü)")
    print(f" 📈 En Yüksek Güven   : %{en_iyi_surucu_kaydi['guven_skoru'] * 100:.2f}")
    print(f" 🎬 Tespit Edilen Kare: Kare {en_iyi_surucu_kaydi['kare_no']}")
    print("-" * 60)
else:
    print("\n❌ Ön cam kırpıntısı içerisinde COCO 'person' nesnesi tespit edilemedi.")

print(f"\n🎥 Analiz videosu kaydedildi: {CIKTI_YOLU}\n" + "=" * 60)

🔄 Modeller entegrasyon için yükleniyor...

🎯 ÖN CAM İÇİ - EN SOLDAKİ KİŞİ FİLTRE TESTİ BAŞLADI...
⏳ Büyük resimdeki diğer insanlar eleniyor, lütfen bekleyin...

🏁 ÖN CAM İÇİ GEOMETRİK ANALİZ TAMAMLANDI!

🏆 ÖN CAM KIRPINTISINDAKİ EN SOLDAN SÜRÜCÜ TESPİT SONUCU:
------------------------------------------------------------
 🎯 Hedef Filtre      : Ön Cam İçi -> En Soldaki Person (Sürücü)
 📈 En Yüksek Güven   : %75.82
 🎬 Tespit Edilen Kare: Kare 156
------------------------------------------------------------

🎥 Analiz videosu kaydedildi: /kaggle/working/on_cam_sol_surucu_sonuc5563.mp4


In [4]:
import cv2
import os
from ultralytics import YOLO

# ==========================================
# 1. DOSYA VE MODEL YOLLARI
# ==========================================
VIDEO_YOLU = '/kaggle/input/datasets/tbitakyandek/telefonkullanyor/video_1 (3).mp4'
CIKTI_YOLU = '/kaggle/working/surucu_full_analizsoninsallah_sonucu.mp4'

# Modellerin Yolları
ONCAM_MODEL_YOLU = '/kaggle/input/models/tbitakyandek/oncambulmamodeli/pytorch/default/1/en_iyi_surucu_tespit_modeli (2).pt'
COCO_MODEL_YOLU  = 'yolov8n.pt' # Sürücüyü (person) ayıklamak için
DAVRANIS_MODEL_YOLU = '/kaggle/input/models/tbitakyandek/sigaratelefonds/pytorch/default/1/best (1).pt' 
YORGUNLUK_MODEL_YOLU = '/kaggle/input/models/tbitakyandek/davranisanalizi/pytorch/default/1/best (4).pt'

# ==========================================
# SINIF İSİMLERİ KONTROLÜ
# ==========================================
TELEFON_ETIKETLERI  = ["phone", "telefon", "cell phone", "talking_phone"]
SIGARA_ETIKETLERI   = ["cigarette", "sigara", "smoking"]
YORGUNLUK_ETIKETLERI = ["yorgun", "fatigue", "yawning", "uyku", "closed_eyes"]

# Modelleri Yükleme
print("🔄 Tüm yapay zeka modelleri pipeline için yükleniyor...")
model_oncam = YOLO(ONCAM_MODEL_YOLU, task='detect')
model_coco = YOLO(COCO_MODEL_YOLU, task='detect')
model_davranis = YOLO(DAVRANIS_MODEL_YOLU, task='detect')
model_yorgunluk = YOLO(YORGUNLUK_MODEL_YOLU, task='detect')

# ==========================================
# 2. VİDEO VE İSTATİSTİK AYARLARI
# ==========================================
cap = cv2.VideoCapture(VIDEO_YOLU)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(CIKTI_YOLU, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# Raporlama için en yüksek güvenli ihlal kayıtları
en_yuksek_ihlal = {
    "telefon": {"guven": 0.0, "kare": 0},
    "sigara": {"guven": 0.0, "kare": 0},
    "yorgunluk": {"durum": "Normal", "guven": 0.0, "kare": 0}
}

# 🔴 CLAHE Nesnesini Oluşturuyoruz (clipLimit ve tileGridSize değerleri araç içi için idealdir)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

print("\n🚀 SÜRÜCÜ MERKEZLİ FULL ANALİZ + CLAHE IŞIK İYİLEŞTİRMESİ BAŞLADI...")
print("⏳ Küçük nesneler (telefon/sigara) için kontrast artırılıyor...\n")

# ==========================================
# 3. PIPELINE DÖNGÜSÜ
# ==========================================
kare_no = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    kare_no += 1

    # ADIM 1: Ön Camı Bul ve Etiketini Çiz kanka
    sonuc_oncam = model_oncam.predict(frame, conf=0.25, verbose=False)

    if len(sonuc_oncam[0].boxes) > 0:
        for cam_box in sonuc_oncam[0].boxes:
            cx1, cy1, cx2, cy2 = map(int, cam_box.xyxy[0])
            cam_guven = float(cam_box.conf[0])
            cam_class_id = int(cam_box.cls[0])
            cam_etiket_adi = model_oncam.names[cam_class_id].upper()
            
            # Ön camı görselleştiriyoruz
            cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), (0, 165, 255), 2)
            cv2.putText(frame, f"{cam_etiket_adi} (%{cam_guven*100:.1f})", (cx1, cy1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
            
            cropped_oncam = frame[cy1:cy2, cx1:cx2]

            # ADIM 2: Ön Cam İçinde Sürücüyü (En Soldaki İnsanı) Seç
            sonuc_coco = model_coco.predict(cropped_oncam, conf=0.25, verbose=False)
            bulunan_insanlar = []

            if len(sonuc_coco[0].boxes) > 0:
                for c_box in sonuc_coco[0].boxes:
                    if int(c_box.cls[0]) == 0: # 0 = Person (COCO)
                        sx1, sy1, sx2, sy2 = map(int, c_box.xyxy[0])
                        bulunan_insanlar.append({"koordinat": (sx1, sy1, sx2, sy2), "sol_x": sx1, "conf": float(c_box.conf[0])})

            if bulunan_insanlar:
                # Soldan sağa sırala ve en soldakini (sürücüyü) al
                bulunan_insanlar.sort(key=lambda x: x["sol_x"])
                suri = bulunan_insanlar[0]
                sx1, sy1, sx2, sy2 = suri["koordinat"]

                # Sürücünün ana resimdeki global koordinatları
                s_ana_x1, s_ana_y1 = cx1 + sx1, cy1 + sy1
                s_ana_x2, s_ana_y2 = cx1 + sx2, cy1 + sy2

                # Güvenli kırpma sınır kontrolü
                s_ana_x1, s_ana_y1 = max(0, s_ana_x1), max(0, s_ana_y1)
                s_ana_x2, s_ana_y2 = min(w, s_ana_x2), min(h, s_ana_y2)
                
                # Sürücü orijinal kırpıntısı
                cropped_surucu = frame[s_ana_y1:s_ana_y2, s_ana_x1:s_ana_x2]
                
                if cropped_surucu.size == 0:
                    continue

                # 🔴 İSTEDİĞİN KRİTİK KISIM: SÜRÜCÜ KIRPINTISINA CLAHE UYGULAMA
                # 1. Görüntüyü gri tonlamaya çeviriyoruz
                gray_surucu = cv2.cvtColor(cropped_surucu, cv2.COLOR_BGR2GRAY)
                # 2. CLAHE algoritmasını çalıştırarak kontrastı dengeliyoruz
                clahe_surucu = clahe.apply(gray_surucu)
                # 3. Model 3 kanallı renkli resim beklediği için tekrar BGR yapıyoruz
                enhanced_surucu = cv2.cvtColor(clahe_surucu, cv2.COLOR_GRAY2BGR)

                # Sürücüyü ana ekranda maviye boyayalım
                cv2.rectangle(frame, (s_ana_x1, s_ana_y1), (s_ana_x2, s_ana_y2), (255, 0, 0), 2)
                cv2.putText(frame, "SURUCU", (s_ana_x1, s_ana_y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

                # --------------------------------------------------
                # ADIM 3: İyileştirilmiş Sürücü (enhanced_surucu) İçinde TELEFON ve SİGARA Arama
                # --------------------------------------------------
                sonuc_davranis = model_davranis.predict(enhanced_surucu, conf=0.25, verbose=False)
                if len(sonuc_davranis[0].boxes) > 0:
                    for d_box in sonuc_davranis[0].boxes:
                        d_id = int(d_box.cls[0])
                        d_isim = model_davranis.names[d_id].lower()
                        d_conf = float(d_box.conf[0])
                        dx1, dy1, dx2, dy2 = map(int, d_box.xyxy[0])

                        # Global koordinata çevir
                        d_ana_x1, d_ana_y1 = s_ana_x1 + dx1, s_ana_y1 + dy1
                        d_ana_x2, d_ana_y2 = s_ana_x1 + dx2, s_ana_y1 + dy2

                        # Telefon İhlali tespiti
                        if any(x in d_isim for x in TELEFON_ETIKETLERI):
                            cv2.rectangle(frame, (d_ana_x1, d_ana_y1), (d_ana_x2, d_ana_y2), (0, 0, 255), 2)
                            cv2.putText(frame, f"TELEFON (%{d_conf*100:.1f})", (d_ana_x1, d_ana_y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                            if d_conf > en_yuksek_ihlal["telefon"]["guven"]:
                                en_yuksek_ihlal["telefon"] = {"guven": d_conf, "kare": kare_no}

                        # Sigara İhlali tespiti
                        elif any(x in d_isim for x in SIGARA_ETIKETLERI):
                            cv2.rectangle(frame, (d_ana_x1, d_ana_y1), (d_ana_x2, d_ana_y2), (0, 165, 255), 2) # Turuncu
                            cv2.putText(frame, f"SIGARA (%{d_conf*100:.1f})", (d_ana_x1, d_ana_y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
                            if d_conf > en_yuksek_ihlal["sigara"]["guven"]:
                                en_yuksek_ihlal["sigara"] = {"guven": d_conf, "kare": kare_no}

                # --------------------------------------------------
                # ADIM 4: İyileştirilmiş Sürücü Üzerinde YORGUNLUK Analizi
                # --------------------------------------------------
                sonuc_yorgunluk = model_yorgunluk.predict(enhanced_surucu, conf=0.25, verbose=False)
                if len(sonuc_yorgunluk[0].boxes) > 0:
                    for y_box in sonuc_yorgunluk[0].boxes:
                        y_id = int(y_box.cls[0])
                        y_isim = model_yorgunluk.names[y_id].lower()
                        y_conf = float(y_box.conf[0])

                        if any(x in y_isim for x in YORGUNLUK_ETIKETLERI):
                            cv2.putText(frame, f"UYARI: YORGUNLUK (%{y_conf*100:.1f})", (s_ana_x1, s_ana_y2 + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                            if y_conf > en_yuksek_ihlal["yorgunluk"]["guven"]:
                                en_yuksek_ihlal["yorgunluk"] = {"durum": y_isim.upper(), "guven": y_conf, "kare": kare_no}

    out.write(frame)

cap.release()
out.release()

# ==========================================
# 5. YAPAY ZEKA PİPELİNE ÖZET RAPORU
# ==========================================
print("=" * 60)
print("🏁 CLAHE DESTEKLİ SÜRÜCÜ FULL ANALİZİ TAMAMLANDI!")
print("=" * 60)
print("\n📊 VİDEO BOYUNCA EN YÜKSEK GÜVENLİ İHLAL VE DURUM RAPORU:")
print("-" * 60)

if en_yuksek_ihlal["telefon"]["guven"] > 0:
    print(f"📱 Telefon Kullanımı : %{en_yuksek_ihlal['telefon']['guven']*100:.2f} güvenle Kare {en_yuksek_ihlal['telefon']['kare']}'de yakalandı.")
else:
    print("📱 Telefon Kullanımı : Temiz / Tespit Edilmedi.")

if en_yuksek_ihlal["sigara"]["guven"] > 0:
    print(f"🚬 Sigara Kullanımı  : %{en_yuksek_ihlal['sigara']['guven']*100:.2f} güvenle Kare {en_yuksek_ihlal['sigara']['kare']}'de yakalandı.")
else:
    print("🚬 Sigara Kullanımı  : Temiz / Tespit Edilmedi.")

if en_yuksek_ihlal["yorgunluk"]["guven"] > 0:
    print(f"💤 Yorgunluk Durumu  : [{en_yuksek_ihlal['yorgunluk']['durum']}] %{en_yuksek_ihlal['yorgunluk']['guven']*100:.2f} güvenle Kare {en_yuksek_ihlal['yorgunluk']['kare']}'de tetiklendi!")
else:
    print("💤 Yorgunluk Durumu  : Sürücü Dinamik / Normal.")

print("-" * 60)
print(f"🎥 Tüm analizlerin işlendiği video çıktısı: {CIKTI_YOLU}\n" + "=" * 60)

🔄 Tüm yapay zeka modelleri pipeline için yükleniyor...

🚀 SÜRÜCÜ MERKEZLİ FULL ANALİZ + CLAHE IŞIK İYİLEŞTİRMESİ BAŞLADI...
⏳ Küçük nesneler (telefon/sigara) için kontrast artırılıyor...

🏁 CLAHE DESTEKLİ SÜRÜCÜ FULL ANALİZİ TAMAMLANDI!

📊 VİDEO BOYUNCA EN YÜKSEK GÜVENLİ İHLAL VE DURUM RAPORU:
------------------------------------------------------------
📱 Telefon Kullanımı : Temiz / Tespit Edilmedi.
🚬 Sigara Kullanımı  : %49.93 güvenle Kare 260'de yakalandı.
💤 Yorgunluk Durumu  : Sürücü Dinamik / Normal.
------------------------------------------------------------
🎥 Tüm analizlerin işlendiği video çıktısı: /kaggle/working/surucu_full_analizsoninsallah_sonucu.mp4
